# Lab 11 - Finetuning in Practice (LoRA on a free Colab T4)

**Chapter 7, "Finetuning"** (memory bottlenecks, memory math, PEFT and LoRA, model merging, finetuning vs RAG) | **Environment:** Google Colab, free **T4 GPU** | **Time:** about 3-4 hours, of which about 25-35 minutes is the GPU running

Last week you built a Larkspur SFT dataset. This week you fine-tune **Qwen3-1.7B** on it with **LoRA** (a cheap way to fine-tune that trains small add-on matrices instead of the whole model) in 16-bit (fp16), on the free Colab T4, using Hugging Face's `transformers` + `peft` + `trl` libraries (for running models, for LoRA, and for training). You will:

- **Part A:** do the memory math for full fine-tuning vs LoRA, choose LoRA **hyperparameters** (settings you pick before training), and run a short training to watch the **loss** (how far the model's answers are from the training answers; lower is better) move.
- **Part B:** run the real fine-tune, evaluate base vs fine-tuned on your 40 held-out examples (ROUGE-L, token F1, and your own reading of 5 examples), compare with your Lab 8 RAG result, save the adapter, and see what merging an adapter does.

**Before you start:**
1. **Runtime > Change runtime type > T4 GPU.** (CPU will not work.)
2. Have your Lab 10 files ready: `larkspur_sft_train.jsonl` and `larkspur_sft_test.jsonl` (in `labs/lab10-dataset-engineering/my_dataset/`). If your Lab 10 dataset did not work out, the notebook downloads the **fallback** dataset automatically.
3. Run cells in order. If the install cell ends with a "Restart session" prompt, click it, then continue with the cell after the install cell.

Colab can't reach the course cluster, and it doesn't need to: everything here runs on the T4.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Fine-tuning, base model** | Fine-tuning is training an existing model a little more on your own examples (from Lab 10). The **base** model is the model before you fine-tune it. | You compare base and fine-tuned answers on the same test questions. |
| **PEFT, LoRA** | PEFT (parameter-efficient fine-tuning) trains only a small number of parameters and freezes the rest. LoRA is the most popular PEFT method: it adds small trainable matrices next to the model's big ones. | Full fine-tuning doesn't fit on a free T4; LoRA does. |
| **Adapter** | The small set of trained LoRA matrices, saved as its own file. | A few MB instead of a few GB; you can keep one base model and swap adapters. |
| **Frozen vs trainable parameters** | Frozen parameters are left as they are; trainable ones are updated during training. | Each trainable parameter costs far more memory than a frozen one (A1). |
| **Hyperparameter** | A setting you choose before training (like the LoRA rank or the learning rate), as opposed to the parameters the model learns. | Bad choices can waste the run or make the model worse. |
| **Rank (r)** | How "thin" the LoRA matrices are. Higher rank means more trainable parameters and more capacity to change the model. | The main LoRA size knob (A2). |
| **Loss** | A number that measures how far the model's predictions are from the training answers. Lower is better. | You watch it to see whether training is working. |
| **Step, batch, epoch** | A batch is a small group of examples processed together. A step is one update of the weights. An epoch is one full pass through the training set. | Training time and the loss curve are counted in steps. |
| **Learning rate** | How big each update to the weights is. | Too high and training becomes unstable; too low and the model barely changes. |
| **Forward/backward pass, gradient, activations** | The forward pass runs examples through the model; the intermediate values it keeps are the **activations**. The backward pass computes a **gradient** for each trainable parameter: which direction, and how much, to change it to lower the loss. | These are what make training need much more memory than inference. |
| **Optimizer (Adam)** | The algorithm that uses the gradients to update the parameters. Adam keeps two extra running averages ("states") per trainable parameter. | Those states are a big part of training memory (A1). |
| **fp16, fp32, mixed precision** | Number formats: 16-bit (2 bytes) and 32-bit (4 bytes). Mixed precision does most of the math in 16-bit but keeps some values in 32-bit for accuracy. | Decides the bytes per parameter in the memory math. |
| **Overfitting** | When a model memorizes its training examples instead of learning the general pattern, so it does worse on new questions. | A low training loss does not guarantee a good model. |
| **`transformers`, `peft`, `trl`** | Hugging Face's Python libraries for running models (`transformers`), for LoRA and other PEFT methods (`peft`), and for training language models (`trl`). | Together they do the whole fine-tune in a few lines. |
| **ROUGE-L, token F1** | Two scores (0 to 1) of how much an answer's wording overlaps with a reference answer. | Cheap to compute with no judge model, but they reward similar wording, not correctness. |
| **Merging** | Folding the adapter back into the base model's weights, giving one normal model again. | No extra speed cost when answering, but you lose the ability to swap adapters. |

## Setup

Versions are pinned (fixed to exact version numbers) so the notebook keeps working when libraries change. Colab's own PyTorch is used as is.

In [ ]:
!pip -q install "itec632 @ git+https://github.com/tyler-schunk/itec632-course#subdirectory=image"
!pip -q install "transformers==5.22.2" "trl==1.17.1" "peft==0.21.3" "accelerate==1.15.0" "datasets==5.1.0" "rouge-score==0.1.2"

In [ ]:
import gc, json, os, re, time, urllib.request
from collections import Counter
from pathlib import Path

import torch, transformers, trl, peft, datasets
import pandas as pd
from itec632 import strip_think
from itec632.grading import LabChecker

ck = LabChecker("lab11", part="colab")
T_START = time.time()
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| datasets", datasets.__version__)

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    gpu = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    gpu = "none"
    print("No GPU! Runtime > Change runtime type > T4 GPU, then run the notebook again.")
ck.check("setup_gpu", HAS_GPU, 1, "Select a T4 GPU runtime (Runtime > Change runtime type).")
ck.record("gpu", gpu)

MODEL_ID = os.environ.get("LAB11_MODEL_ID", "Qwen/Qwen3-1.7B")   # (instructor testing hook; leave as is)
USE_FP16 = HAS_GPU            # the T4 has no bfloat16, so we train in float16 with mixed precision
DTYPE = torch.float16 if HAS_GPU else torch.float32

### Load your Lab 10 dataset

Upload `larkspur_sft_train.jsonl` and `larkspur_sft_test.jsonl` with the **Files** panel on the left (folder icon > upload). If they are not there (or you set `USE_FALLBACK = True`), the fallback dataset from the course repo is used instead. Say which one you used in your analysis.

In [ ]:
USE_FALLBACK = False
TRAIN_FILE, TEST_FILE = Path("larkspur_sft_train.jsonl"), Path("larkspur_sft_test.jsonl")
if USE_FALLBACK or not (TRAIN_FILE.exists() and TEST_FILE.exists()):
    print("Using the FALLBACK dataset (your Lab 10 files were not found or USE_FALLBACK=True).")
    BASE_URL = "https://raw.githubusercontent.com/tyler-schunk/itec632-course/main/labs/lab10-dataset-engineering/fallback_dataset/"
    TRAIN_FILE, TEST_FILE = Path("FALLBACK_larkspur_sft_train.jsonl"), Path("FALLBACK_larkspur_sft_test.jsonl")
    for f in (TRAIN_FILE, TEST_FILE):
        if not f.exists():
            urllib.request.urlretrieve(BASE_URL + f.name, f)

def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]

train_rows, test_rows = read_jsonl(TRAIN_FILE), read_jsonl(TEST_FILE)
DATASET = "fallback" if "FALLBACK" in TRAIN_FILE.name else "own"
print(f"{DATASET} dataset: {len(train_rows)} train, {len(test_rows)} test")
print(Counter(r.get("meta", {}).get("source_type", "?") for r in train_rows))
print(json.dumps(train_rows[0]["messages"], indent=1)[:700])

ok_format = all([m["role"] for m in r["messages"]][-1] == "assistant" for r in train_rows + test_rows)
ck.check("data_loaded", 100 <= len(train_rows) <= 600 and len(test_rows) == 40 and ok_format, 2,
         "Expect 200-400 chat-format training examples and exactly 40 test examples, each ending with an assistant turn.")
ck.record("dataset", DATASET)
ck.record("n_train", len(train_rows))

**Reading the output:** which dataset you're using and how many training and test examples it has; a count of training examples by source type (`doc_qa` = generated from a document, `ticket_note` = from a real ticket); and the first training example in chat format, a list of messages with roles (`system`, `user`, `assistant`). The `assistant` message is the answer the model learns to give.

## Part A - Guided practice

### A1. Memory math: why full fine-tuning doesn't fit on a T4

The chapter's "Memory Bottlenecks" section ("Memory Math") breaks training memory into **weights + activations + gradients + optimizer states**. Here is what those are. Training repeats two passes over a small group of examples (a **batch**). The **forward pass** runs the examples through the model and keeps its intermediate results (**activations**). The **backward pass** uses them to compute a **gradient** for every trainable parameter: which direction, and how much, to change it to lower the loss. Then the **optimizer** updates the parameters; the Adam optimizer keeps two extra running averages (its "states") for every trainable parameter to do that well. Inference only needs the weights (plus the KV cache), which is the math you did in Lab 2: parameters x bytes per parameter.

For training with the Adam optimizer in **mixed precision** (most math in 16-bit fp16, some values kept in 32-bit fp32 for accuracy), a common rule of thumb **per trainable parameter** is:

| what | bytes |
|---|---|
| fp16 weight (used in forward/backward) | 2 |
| fp16 gradient | 2 |
| fp32 master copy of the weight | 4 |
| Adam first moment (m), fp32 | 4 |
| Adam second moment (v), fp32 | 4 |
| **total** | **16** |

(The chapter's "Memory needed for training" example is simpler: one gradient value plus two Adam states per trainable parameter, at 2 bytes each, so 13B x 3 x 2 bytes = 78 GB on top of the weights. The 16-byte rule above is what you get in mixed-precision training, where the master weights and Adam states are kept in fp32. Either way, the point is the same.)

A **frozen** parameter (not trained) only needs its 2-byte weight. That is the whole trick behind parameter-efficient fine-tuning (**PEFT**, methods that train only a small fraction of the parameters): freeze almost everything.

Load the model and tokenizer (about 1-2 minutes the first time).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load_base():
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="auto" if HAS_GPU else None)
    m.config.use_cache = False          # caching is for generation; it wastes memory during training
    return m

model = load_base()
n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_ID}: {n_params / 1e9:.3f} B parameters, dtype {next(model.parameters()).dtype}")
if HAS_GPU:
    print(f"GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

**TODO (A1):** compute, in GB (1 GB = 1e9 bytes, ignore activations):
- `inference_gb`: weights only, 2 bytes per parameter.
- `full_ft_gb`: every parameter trainable, 16 bytes per parameter.

In [ ]:
inference_gb = None   # TODO
full_ft_gb = None     # TODO

T4_GB = 15.0
if inference_gb and full_ft_gb:
    print(f"inference (weights only): {inference_gb:.1f} GB | full fine-tuning (no activations): {full_ft_gb:.1f} GB | T4: {T4_GB} GB")

In [ ]:
ck.check("A1_memory_math", inference_gb is not None and full_ft_gb is not None
         and abs(inference_gb - n_params * 2 / 1e9) < 0.05 and abs(full_ft_gb - n_params * 16 / 1e9) < 0.2, 2,
         "inference_gb = n_params * 2 / 1e9; full_ft_gb = n_params * 16 / 1e9.")
ck.record("n_params_B", round(n_params / 1e9, 3))
ck.record("full_ft_gb", round(full_ft_gb or 0, 1))

### A2. LoRA hyperparameters

LoRA (chapter: "LoRA" under "Finetuning Techniques") freezes the model and learns, for selected weight matrices W, a low-rank update W + (alpha / r) * B A, where A and B are thin matrices of rank r. In plain words: instead of changing a big weight matrix directly, LoRA learns two small matrices whose product is a correction added to it. The rank r says how thin they are: a 2,048 x 2,048 matrix holds about 4.2 million numbers, but with r = 16 the two LoRA matrices hold only 2 x 2,048 x 16, about 65,000. These small trained matrices are the **adapter**. Three choices matter most:

- **`r` (rank):** capacity of the update. The chapter notes that small ranks (4-64) usually work; higher costs more memory and can **overfit** a small dataset (memorize the training examples instead of learning the general pattern, so it does worse on new questions).
- **`lora_alpha`:** scales the update (the effective scale is alpha / r). The chapter says the alpha:r ratio is typically between 1:8 and 8:1; alpha = 2r is a common starting point.
- **`target_modules`:** which matrices get adapters. The chapter notes LoRA is most commonly applied to the four attention matrices (query, key, value, output: `q/k/v/o_proj`), that the LoRA authors found query and value to be the best pair if you can pick only two, and that applying LoRA to more matrices, including the feedforward (MLP `gate/up/down_proj`) layers, tends to work better. We use all seven for a small cost.

Print the module names to see what's available (a **module** is a named piece of the model; here we list the kinds of weight matrices inside each layer):

In [ ]:
linear_names = sorted({name.split(".")[-1] for name, mod in model.named_modules() if isinstance(mod, torch.nn.Linear)})
print(linear_names)

**Reading the output:** `q_proj`, `k_proj`, `v_proj`, and `o_proj` are attention's query, key, value, and output matrices (Lab 02); `gate_proj`, `up_proj`, and `down_proj` are the MLP's three matrices. `lm_head` is the output layer that turns each final vector into a score for every token.

**TODO (A2):** create `lora_config = LoraConfig(...)` with `r=16`, `lora_alpha=32`, `lora_dropout=0.05` (randomly ignores 5% of the adapter's inputs during training, which helps against overfitting), `bias="none"` (don't train the small bias values), `task_type="CAUSAL_LM"` (a model that predicts the next token, like every model in this course), and `target_modules` set to all seven projection layers above (not `lm_head`). Then the cell wraps the model and counts trainable parameters. (You may try other values later; explain any change in your analysis.)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = None   # TODO: LoraConfig(r=..., lora_alpha=..., lora_dropout=..., bias="none", task_type="CAUSAL_LM", target_modules=[...])

if lora_config is None:
    print("Complete the TODO above first. Using r=8 on q_proj/v_proj for now so the notebook keeps running.")
    lora_config_used = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
else:
    lora_config_used = lora_config

In [ ]:
model = get_peft_model(model, lora_config_used)     # adapters are created in fp32 even though the base is fp16
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
lora_gb = (n_params * 2 + trainable * 16) / 1e9
print(f"trainable: {trainable / 1e6:.2f} M of {n_params / 1e9:.3f} B ({100 * trainable / n_params:.2f}%)")
print(f"LoRA memory estimate (no activations): {lora_gb:.2f} GB vs full fine-tuning {full_ft_gb or 0:.1f} GB")

tm = set(getattr(lora_config, "target_modules", None) or [])
ck.check("A2_lora_config", lora_config is not None and 4 <= lora_config.r <= 64 and {"q_proj", "v_proj"} <= tm
         and "lm_head" not in tm, 2, "Create a LoraConfig with r between 4 and 64 that targets at least q_proj and v_proj (not lm_head).")
ck.check("A2_trainable_fraction", 0 < trainable / n_params < 0.05, 1, "LoRA should train well under 5% of the parameters.")
ck.record("lora_r", lora_config_used.r)
ck.record("lora_alpha", lora_config_used.lora_alpha)
ck.record("lora_targets", sorted(lora_config_used.target_modules))
ck.record("trainable_M", round(trainable / 1e6, 2))
ck.record("lora_gb_estimate", round(lora_gb, 2))

### A3. Train a few steps and watch the loss

**Reading the output above:** `trainable` is the number of adapter parameters and the share of the whole model they make up. The memory estimate applies your A1 rule: 2 bytes for every frozen weight plus 16 bytes for every trainable one.

SFT data is converted to **prompt-completion** form: the prompt is the system + user turns, the completion is the assistant turn. TRL then computes the **loss** (for each token of the answer, how little probability the model gave the right token, averaged) **only on the completion**, so the model learns to produce answers, not to repeat the question. `chat_template_kwargs` turns off Qwen3's thinking mode, matching how we will generate later.

Then a 15-step run (about a minute on a T4). A **step** is one update of the weights. In `sft_args`, the model looks at 4 examples at a time (`per_device_train_batch_size`) and adds up the gradients of 2 such batches before each update (`gradient_accumulation_steps`), so each step uses 8 examples. `learning_rate` sets how big each update is. Look at the loss: it should drop quickly at first.

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def to_prompt_completion(row):
    m = row["messages"]
    return {"prompt": m[:-1], "completion": m[-1:], "chat_template_kwargs": {"enable_thinking": False}}

train_ds = Dataset.from_list([to_prompt_completion(r) for r in train_rows])
print(train_ds)

def sft_args(**overrides):
    base = dict(output_dir="runs", per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=5, logging_steps=5,
                max_length=1024, fp16=USE_FP16, bf16=False, gradient_checkpointing=True,
                gradient_checkpointing_kwargs={"use_reentrant": False}, optim="adamw_torch",
                save_strategy="no", report_to="none", seed=632)
    base.update(overrides)
    return SFTConfig(**base)

tok.padding_side = "right"
short = SFTTrainer(model=model, args=sft_args(max_steps=15, logging_steps=1), train_dataset=train_ds, processing_class=tok)
t0 = time.time()
short.train()
short_losses = [h["loss"] for h in short.state.log_history if "loss" in h]
print(f"{len(short_losses)} logged losses in {time.time() - t0:.0f}s:", [round(x, 3) for x in short_losses])

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3))
plt.plot(range(1, len(short_losses) + 1), short_losses, marker="o")
plt.xlabel("step"); plt.ylabel("training loss"); plt.title("A3: first 15 steps"); plt.tight_layout(); plt.show()

ck.check("A3_loss_logged", len(short_losses) >= 10, 1, "The short run should log a loss every step (logging_steps=1).")
first, last = (sum(short_losses[:3]) / 3, sum(short_losses[-3:]) / 3) if len(short_losses) >= 6 else (0, 0)
ck.check("A3_loss_decreases", len(short_losses) >= 6 and last < first, 2,
         "The average of the last 3 losses should be below the first 3. If not, check the learning rate and the data.")
ck.record("short_run_first_loss", round(first, 3))
ck.record("short_run_last_loss", round(last, 3))

# free the GPU before the real run: we start Part B from a fresh copy of the base model
del short, model
gc.collect()
if HAS_GPU:
    torch.cuda.empty_cache()

**Reading the plot:** across is the training step (1-15), up is the training loss on that step's batch. The line is bumpy because each step sees different examples; look at the overall trend.

## Part B - Applied lab

### B1. Baseline: the base model on your 40 test examples

Before training, record what the **base** model does on the held-out test set. Generation is greedy (no sampling) so base and fine-tuned outputs are comparable, and batched with left padding so 40 examples take a minute or two. (Batched: 8 examples run together. Shorter prompts are filled out with padding tokens on the left, so all prompts end at the same place, where the answer starts.)

In [ ]:
def generate(model, rows, max_new_tokens=200, batch_size=8):
    """Greedy answers for the prompt (all turns but the last) of each test row."""
    model.eval()
    old_side, tok.padding_side = tok.padding_side, "left"
    outs = []
    for i in range(0, len(rows), batch_size):
        prompts = [tok.apply_chat_template(r["messages"][:-1], tokenize=False, add_generation_prompt=True,
                                           enable_thinking=False) for r in rows[i:i + batch_size]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += [strip_think(t) for t in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    tok.padding_side = old_side
    return outs

base = load_base()
base.config.use_cache = True
t0 = time.time()
base_preds = generate(base, test_rows)
print(f"{len(base_preds)} base answers in {time.time() - t0:.0f}s\n")
print("Q:", test_rows[0]["messages"][1]["content"][:300], "\nBASE:", base_preds[0][:300], "\nREF:", test_rows[0]["messages"][2]["content"][:300])
ck.check("B1_base_predictions", len(base_preds) == len(test_rows) and sum(bool(p.strip()) for p in base_preds) >= 30, 1,
         "generate() should return a non-empty answer for (almost) every test example.")

**Reading the output:** `Q` is the first test question, `BASE` the base model's answer, and `REF` the reference answer from your test set (the answer you want).

### B2. The real fine-tune

Same recipe as A3, on the whole training set for 2 **epochs** (an epoch is one full pass through the training set; about 60-70 optimizer steps for 250 examples; roughly 5-10 minutes on a T4). We track **peak GPU memory** to compare with your A1/A2 estimates.

**TODO (B2):** set `NUM_EPOCHS` (start with 2) and keep `LEARNING_RATE = 2e-4` unless you have a reason to change it. If you change either, say why in Q2.

In [ ]:
NUM_EPOCHS = None        # TODO: 2 is a good start
LEARNING_RATE = 2e-4

In [ ]:
NUM_EPOCHS = NUM_EPOCHS or 1
base.config.use_cache = False
model = get_peft_model(base, lora_config_used)
if HAS_GPU:
    torch.cuda.reset_peak_memory_stats()
tok.padding_side = "right"
trainer = SFTTrainer(model=model, args=sft_args(num_train_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE),
                     train_dataset=train_ds, processing_class=tok)
result = trainer.train()
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if HAS_GPU else 0.0
losses = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
print(f"train_runtime {result.metrics['train_runtime']:.0f}s | steps {result.global_step} | "
      f"final loss {losses[-1][1] if losses else float('nan'):.3f} | peak GPU memory {peak_gb:.2f} GB")

plt.figure(figsize=(6, 3))
plt.plot([s for s, _ in losses], [l for _, l in losses], marker=".")
plt.xlabel("step"); plt.ylabel("training loss"); plt.title("B2: full run"); plt.tight_layout(); plt.show()

ck.check("B2_training_finished", len(losses) >= 3 and losses[-1][1] < losses[0][1], 2,
         "Training should finish and the last logged loss should be below the first.")
ck.check("B2_peak_memory_recorded", (0 < peak_gb < 15) or not HAS_GPU, 1, "Peak memory should be well under the T4's 15 GB.")
ck.record("epochs", NUM_EPOCHS)
ck.record("train_steps", int(result.global_step))
ck.record("train_runtime_s", round(result.metrics["train_runtime"], 1))
ck.record("final_train_loss", round(losses[-1][1], 3) if losses else None)
ck.record("peak_gpu_gb", round(peak_gb, 2))

**Reading the output:** the printed line gives the training time in seconds, the number of steps, the last logged loss, and the most GPU memory used at any moment (peak). In the plot, across is the step and up is the training loss, logged every 5 steps.

### B3. Evaluate: base vs fine-tuned

Two judge-free metrics against your reference answers (Ch. 3 "similarity measurements against reference data"):
- **ROUGE-L:** overlap of the longest common subsequence of words (the longest run of words that appear in both answers in the same order, not necessarily next to each other).
- **Token F1:** overlap of the bags of words (the words and their counts, ignoring order), as in SQuAD (a well-known question-answering benchmark). **Precision** is the share of the answer's words that are in the reference; **recall** is the share of the reference's words that the answer contains; F1 combines the two.

Both reward wording close to the reference, not correctness as such; that's why you also read 5 examples yourself. We report the metrics separately for **doc Q&A** (facts) and **ticket notes** (form).

**TODO (B3):** finish `token_f1(pred, ref)`: count the overlap with `Counter(p) & Counter(r)`; return 0.0 if the overlap is 0; otherwise precision = overlap / len(p), recall = overlap / len(r), and F1 = 2PR / (P + R).

In [ ]:
from rouge_score import rouge_scorer

def words(s):
    return re.findall(r"\w+", s.lower())

def token_f1(pred, ref):
    p, r = words(pred), words(ref)
    if not p or not r:
        return 0.0
    # TODO: overlap = sum((Counter(p) & Counter(r)).values()); compute precision, recall, F1
    return 0.0

print(token_f1("reset the password and verified by callback", "verified by callback then reset password"))  # about 0.77

In [ ]:
model.config.use_cache = True
t0 = time.time()
ft_preds = generate(model, test_rows)
print(f"{len(ft_preds)} fine-tuned answers in {time.time() - t0:.0f}s")

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rows = []
for r, b, f in zip(test_rows, base_preds, ft_preds):
    ref = r["messages"][-1]["content"]
    for name, pred in (("base", b), ("finetuned", f)):
        rows.append({"source_type": r.get("meta", {}).get("source_type", "?"), "model": name,
                     "rougeL": scorer.score(ref, pred)["rougeL"].fmeasure, "token_f1": token_f1(pred, ref),
                     "pred_words": len(words(pred))})
scores = pd.DataFrame(rows)
table = scores.groupby(["source_type", "model"])[["rougeL", "token_f1", "pred_words"]].mean().round(3)
display(table)
overall = scores.groupby("model")[["rougeL", "token_f1"]].mean().round(3)
display(overall)

ck.check("B3_token_f1", abs(token_f1("reset the password and verified by callback",
                                     "verified by callback then reset password") - 0.769) < 0.01, 1,
         "token_f1 should be about 0.769 for the example above.")
ck.check("B3_metrics_computed", len(scores) == 2 * len(test_rows), 1, "Score every test example for both models.")
ft_better = overall.loc["finetuned", "rougeL"] > overall.loc["base", "rougeL"] if len(overall) == 2 else False
ck.check("B3_finetune_improves", bool(ft_better), 2,
         "Usually the fine-tuned model scores higher overall. If not, check that training finished and the loss dropped.")
for (st, m), v in table.iterrows():
    ck.record(f"rougeL_{st}_{m}", float(v["rougeL"]))
    ck.record(f"tokenF1_{st}_{m}", float(v["token_f1"]))

**Reading the tables:** the first table has one row per source type and model (`base` or `finetuned`): the average ROUGE-L and token F1 (both 0 to 1; higher means closer to the reference wording) and `pred_words`, the average answer length in words. The second table averages over all 40 test examples.

Now read. The cell shows 5 test examples (doc Q&A and ticket notes mixed): question, reference, base answer, fine-tuned answer.

**TODO (B3b):** for each, write one line in `MANUAL_NOTES`: which answer is better and why (correct facts? Larkspur-specific? right format? made something up?).

In [ ]:
pick = [i for i, r in enumerate(test_rows) if r.get("meta", {}).get("source_type") == "doc_qa"][:3] + \
       [i for i, r in enumerate(test_rows) if r.get("meta", {}).get("source_type") == "ticket_note"][:2]
pick = (pick + list(range(len(test_rows))))[:5] if len(pick) < 5 else pick
for n, i in enumerate(pick):
    r = test_rows[i]
    print(f"===== [{n}] {r.get('meta', {}).get('source_type')} ({r.get('meta', {}).get('source_id')})")
    print("Q:   ", r["messages"][1]["content"][:400])
    print("REF: ", r["messages"][2]["content"][:400])
    print("BASE:", base_preds[i][:400])
    print("FT:  ", ft_preds[i][:400], "\n")

In [ ]:
MANUAL_NOTES = {
    0: "",   # TODO: e.g. "FT better: gives the 2026 price ($145); base invents a generic MSP price range"
    1: "",
    2: "",
    3: "",
    4: "",
}

In [ ]:
ck.check("B3_manual_notes", sum(len(str(v).strip()) >= 15 for v in MANUAL_NOTES.values()) >= 5, 1,
         "Write a one-line judgment (15+ characters) for each of the 5 examples.")

### B4. Finetuning vs RAG

The chapter's "Finetuning and RAG" discussion: finetuning tends to fix **form** (behavior, style, format), while RAG supplies **facts** (information the model doesn't have, or that changes). Your table above lets you check that on your own data. If you have it, paste your Lab 8 RAG correctness (the fraction of questions your RAG pipeline answered correctly) for comparison. Different metric, different questions, so treat it as a rough reference point and say so.

In [ ]:
LAB8_RAG_CORRECTNESS = None   # optional: e.g. 0.82 from your Lab 8 notebook
ck.record("lab8_rag_correctness", LAB8_RAG_CORRECTNESS)
doc_ft = table.loc[("doc_qa", "finetuned"), "token_f1"] if ("doc_qa", "finetuned") in table.index else None
note_gain = None
if ("ticket_note", "finetuned") in table.index and ("ticket_note", "base") in table.index:
    note_gain = table.loc[("ticket_note", "finetuned"), "rougeL"] - table.loc[("ticket_note", "base"), "rougeL"]
qa_gain = None
if ("doc_qa", "finetuned") in table.index and ("doc_qa", "base") in table.index:
    qa_gain = table.loc[("doc_qa", "finetuned"), "rougeL"] - table.loc[("doc_qa", "base"), "rougeL"]
print(f"ROUGE-L gain from fine-tuning: ticket notes {note_gain}, doc Q&A {qa_gain}")
ck.record("rougeL_gain_ticket_note", None if note_gain is None else round(float(note_gain), 3))
ck.record("rougeL_gain_doc_qa", None if qa_gain is None else round(float(qa_gain), 3))

### B5. Save the adapter, and what merging does

A LoRA adapter is tiny compared with the model: you can keep one base model and swap in many adapters (one per task or client), which the chapter calls **multi-LoRA serving** (in the "LoRA" section). **Merging** (`merge_and_unload`) folds W + (alpha/r) B A back into W: the result is a normal model with no extra inference latency, but it is full size again and you lose the ability to swap adapters. (The chapter's "Model Merging and Multi-Task Finetuning" section goes further: merging *several* fine-tuned models into one.)

In [ ]:
ADAPTER_DIR = Path("larkspur-lora")
model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
adapter_mb = sum(f.stat().st_size for f in ADAPTER_DIR.glob("adapter_model*")) / 1e6
base_mb = n_params * 2 / 1e6
print(sorted(p.name for p in ADAPTER_DIR.iterdir()))
print(f"adapter: {adapter_mb:.1f} MB vs base model weights (fp16): {base_mb:,.0f} MB ({100 * adapter_mb / base_mb:.2f}%)")

probe = [test_rows[pick[0]]]
before = generate(model, probe, max_new_tokens=60)[0]
merged = model.merge_and_unload()          # in place: W <- W + (alpha/r) * B @ A
merged_params = sum(p.numel() for p in merged.parameters())
after = generate(merged, probe, max_new_tokens=60)[0]
same = scorer.score(before, after)["rougeL"].fmeasure if before and after else 0.0
print(f"merged model parameters: {merged_params / 1e9:.3f} B (base had {n_params / 1e9:.3f} B)")
print(f"adapter output: {before[:150]!r}\nmerged output:  {after[:150]!r}\nsimilarity: {same:.2f}")

ck.check("B5_adapter_saved", (ADAPTER_DIR / "adapter_config.json").exists() and 0 < adapter_mb < 300, 2,
         "save_pretrained should write adapter_config.json and an adapter_model file of a few tens of MB.")
ck.check("B5_merge", merged_params == n_params and same >= 0.8, 1,
         "After merge_and_unload the parameter count equals the base model's and outputs stay (nearly) the same.")
ck.record("adapter_mb", round(adapter_mb, 1))
ck.record("merge_output_similarity", round(float(same), 2))
ck.record("notebook_minutes", round((time.time() - T_START) / 60, 1))

**Reading the output:** the saved adapter folder holds `adapter_config.json` (the LoRA settings) and `adapter_model.safetensors` (the trained numbers). Next, the adapter's size compared with the base model's weights. Then the merged model's parameter count, and one answer generated before and after merging, with a ROUGE-L similarity between them (1.00 means identical).

Optional: download the adapter to keep it (Colab deletes files when the session ends).

In [ ]:
DOWNLOAD_ADAPTER = False
if DOWNLOAD_ADAPTER:
    import shutil
    shutil.make_archive("larkspur-lora", "zip", ADAPTER_DIR)
    try:
        from google.colab import files
        files.download("larkspur-lora.zip")
    except ImportError:
        print("Not on Colab: larkspur-lora.zip is in the current folder.")

## Written analysis (40 points)

Use your own numbers from this notebook (the summary at the end lists them) and cite Chapter 7 by section name or concept.

**Q1. Memory bottlenecks.** Compare your A1/A2 estimates (full fine-tuning, LoRA) with the peak GPU memory you measured in B2. What explains the gap between the LoRA estimate and the measured peak? (Think about what the 16-bytes rule leaves out: activations, sequence length, batch size, the logits over a ~150k-token vocabulary.) Which knobs in `sft_args` would you turn first if you ran out of memory, and why does gradient checkpointing trade compute for memory?

*Your answer:*

**Q2. LoRA choices and the loss.** Report your LoRA config (r, alpha, target modules), trainable parameter count, epochs, and how the loss behaved in A3 and B2. Using the chapter's discussion of LoRA hyperparameters, what would you expect from r=4 vs r=64 on a ~250-example dataset? Does a low final training loss mean a good model? (Look at your test metrics.)

*Your answer:*

**Q3. Results: facts vs form.** Using the B3 table, compare the gains for doc Q&A and ticket notes, and use your 5 manual notes. Did fine-tuning help more with form or with facts? Did it introduce any confident wrong facts? Were your Lab 10 predictions (Q4 there) right?

*Your answer:*

**Q4. Finetune or RAG for Larkspur?** Using the chapter's "When to Finetune" and "Finetuning and RAG" sections, your results, and (if you have it) your Lab 8 RAG number: for the help-desk assistant, what would you finetune, what would you leave to RAG, and why? Consider that Larkspur's prices and procedures change every year (what happens to a fine-tuned model when they do?), and how adapter size and merging affect deployment.

*Your answer:*

## AI-use reflection (10 points)

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. Fine-tuning libraries change fast (this notebook pins versions for that reason). Did an AI assistant suggest any code for this lab that used an outdated or non-existent API (for example an old argument name)? How did you find out?

*Your answer:*

## Submit

Run this last, save (**File > Download > Download .ipynb**), and upload the .ipynb to Canvas.

In [ ]:
ck.report()